In [2]:
import pandas as pd
import xarray as xr
import netCDF4 as nc
import os

Sara Hamilton - June 9, 2026. 
This code is based off of Faith Townsend's Live_Ocean_S&T_wrangling code. It takes .nc files of Live Ocean ROMS
data and breaks them into .csv files that contain the surface ocean data needed to analyze nearshore Oregon 
conditions from 2013-2023. 


In [27]:
# Load .nc Live Ocean ROMS dataset
# Do this once for the 2013-2019 data. Then go change the ds filename and the years to do it for the 2020-2024 years
# Live Ocean Data was provided to us in two .nc files, one covering 2013-2019 and one convering 2020-2024. 
ds = xr.open_dataset(r"D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/townsend2_surf_2020.01.01_2024.12.31.nc",
                    engine="netcdf4")
# Ensure ocean_time is in datetime format
ds['ocean_time'] = pd.to_datetime(ds['ocean_time'].values)
output_dir = "D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_datasets/"
os.makedirs(output_dir, exist_ok=True)  # Ensure output directory exists
#modifying this to just work with temperature for now. 
# Parker McCready at Live Ocean provided us data files that only contain surface data. Normally would need to specify what depths you want to pull. 
#So I modified the indices to not pull from a fourth depth index
years = [2020,2021,2022,2023,2024] # OR USE [2013, 2014, 2015, 2016, 2017, 2018, 2019]
for year in years:
    for month in range(1, 13):  # Loop over months from January (1) to December (12)
        # Subset dataset for the specific year and month
        monthly_ds = ds.sel(ocean_time=(ds['ocean_time'].dt.year == year) & (ds['ocean_time'].dt.month == month))
        required_vars = [ 'hc', 'h', 'zeta', 'temp', 'lon_rho', 'lat_rho']

        if all(var in monthly_ds.variables for var in required_vars):
                      # Extract surface temperature (assuming -1 index gives the topmost layer)
            surf_temp = monthly_ds.temp[:, :, :].to_numpy()  # Extract top layer over time
            
            # Convert to xarray DataArray to preserve metadata
            surf_temp_da = xr.DataArray(surf_temp, dims=["ocean_time", "eta_rho", "xi_rho"], 
                                  coords={"ocean_time": monthly_ds.ocean_time, 
                                        "lat_rho": monthly_ds.lat_rho, 
                                        "lon_rho": monthly_ds.lon_rho},
                                name="surf_temp")
            
            temp_ds = xr.Dataset({
                "ocean_time": monthly_ds.ocean_time,
                "lat_rho": monthly_ds.lat_rho,
                "lon_rho": monthly_ds.lon_rho,
                "surf_temp": surf_temp_da
            })
            
            # Save datasets to disk
            temp_filename = f"{output_dir}ds_temp_{year}_{month:02d}.nc"
            temp_ds.to_netcdf(temp_filename)
            
            del temp_ds  # Free up memory 

In [30]:
#Ok step two, moving data out of .ncs and into csvs
#Do it once with the years =[2013, 2014, 2015, 2016, 2017, 2018, 2019] and onces as [2020,2021,2022,2023,2024]
# Define directories
input_dir = "D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_datasets"
output_dir = "D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_datasets_csvs"

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# Define the years and months to loop over
years = [2020,2021,2022,2023,2024]
months = range(1, 13)  # January (1) to December (12)

# Process each dataset one by one
for year in years:
    for month in months:
        # Update the filename for salt and temp datasets
        filename_temp = f"ds_temp_{year}_{month:02d}.nc"
        
        # Construct file paths for salt and temp datasets
        file_path_temp = os.path.join(input_dir, filename_temp)

        # Check if temp file exists and process it
        if os.path.exists(file_path_temp):
            print(f"Processing {filename_temp}...")
            ds_temp = xr.open_dataset(file_path_temp)
            df_temp = ds_temp.to_dataframe().reset_index()
            output_filename_temp = f"surface_temp_{year}_{month:02d}.csv"
            output_path_temp = os.path.join(output_dir, output_filename_temp)
            df_temp.to_csv(output_path_temp, index=False)
            ds_temp.close()
            del ds_temp, df_temp
            print(f"Saved {output_filename_temp}")

print("Processing complete. All surface data saved.")

Processing ds_temp_2020_01.nc...
Saved surface_temp_2020_01.csv
Processing ds_temp_2020_02.nc...
Saved surface_temp_2020_02.csv
Processing ds_temp_2020_03.nc...
Saved surface_temp_2020_03.csv
Processing ds_temp_2020_04.nc...
Saved surface_temp_2020_04.csv
Processing ds_temp_2020_05.nc...
Saved surface_temp_2020_05.csv
Processing ds_temp_2020_06.nc...
Saved surface_temp_2020_06.csv
Processing ds_temp_2020_07.nc...
Saved surface_temp_2020_07.csv
Processing ds_temp_2020_08.nc...
Saved surface_temp_2020_08.csv
Processing ds_temp_2020_09.nc...
Saved surface_temp_2020_09.csv
Processing ds_temp_2020_10.nc...
Saved surface_temp_2020_10.csv
Processing ds_temp_2020_11.nc...
Saved surface_temp_2020_11.csv
Processing ds_temp_2020_12.nc...
Saved surface_temp_2020_12.csv
Processing ds_temp_2021_01.nc...
Saved surface_temp_2021_01.csv
Processing ds_temp_2021_02.nc...
Saved surface_temp_2021_02.csv
Processing ds_temp_2021_03.nc...
Saved surface_temp_2021_03.csv
Processing ds_temp_2021_04.nc...
Saved s

In [3]:
#Ok this step should let you knit all the csvs together into one, which should be integrable with the larger surfaces dataframes
in_dir = "D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_datasets_csvs"
out_dir = "D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_yearlyfiles"

# Define the years to process
years = [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

# Loop through each year and combine salt and temp datasets separately
for year in years:
    # Get list of CSV files for salt and temp data
    temp_files = [f for f in os.listdir(in_dir) if f.startswith(f"surface_temp_{year}_") and f.endswith(".csv")]

    # Initialize lists to store dataframes
    df_temp_list = []

    # Load and append temp data
    for file in temp_files:
        file_path = os.path.join(in_dir, file)
        print(f"Loading {file} (Temp)...")
        df_temp_list.append(pd.read_csv(file_path))

    # Combine all temp data for the year and save
    if df_temp_list:
        df_temp_combined = pd.concat(df_temp_list, ignore_index=True)
        temp_output_path = os.path.join(out_dir, f"surface_temp_{year}_combined.csv")
        df_temp_combined.to_csv(temp_output_path, index=False)
        print(f"Combined temp data saved to {temp_output_path}")

print("Processing complete. All combined surface data saved.")

Loading surface_temp_2014_01.csv (Temp)...
Loading surface_temp_2014_02.csv (Temp)...
Loading surface_temp_2014_03.csv (Temp)...
Loading surface_temp_2014_04.csv (Temp)...
Loading surface_temp_2014_05.csv (Temp)...
Loading surface_temp_2014_06.csv (Temp)...
Loading surface_temp_2014_07.csv (Temp)...
Loading surface_temp_2014_08.csv (Temp)...
Loading surface_temp_2014_09.csv (Temp)...
Loading surface_temp_2014_10.csv (Temp)...
Loading surface_temp_2014_11.csv (Temp)...
Loading surface_temp_2014_12.csv (Temp)...
Combined temp data saved to D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_yearlyfiles\surface_temp_2014_combined.csv
Loading surface_temp_2015_01.csv (Temp)...
Loading surface_temp_2015_02.csv (Temp)...
Loading surface_temp_2015_03.csv (Temp)...
Loading surface_temp_2015_04.csv (Temp)...
Loading surface_temp_2015_05.csv (Temp)...
Loading surface_temp_2015_06.csv (Temp)...
Loading surface_temp_2015_07.csv (Temp)...
Loading surface_temp_2015_08.csv (Temp)...


In [2]:
# Define directories
data_dir = "D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_yearlyfiles"

# Define the years to process
years = [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

# Initialize lists to store dataframes for temp data
df_temp_list = []

# Loop through each year and load temp datasets
for year in years:
    # Get list of CSV files for salt and temp data
    temp_files = [f for f in os.listdir(data_dir) if f.startswith(f"surface_temp_{year}_") and f.endswith("combined.csv")]

    # Load and append temp data
    for file in temp_files:
        file_path = os.path.join(data_dir, file)
        print(f"Loading {file} (Temp)...")
        df_temp_list.append(pd.read_csv(file_path))

# Combine all temp data into one file and save
if df_temp_list:
    df_temp_combined = pd.concat(df_temp_list, ignore_index=True)
    temp_output_path = os.path.join(data_dir, "surface_temp_all.csv")
    df_temp_combined.to_csv(temp_output_path, index=False)
    print(f"Combined temp data saved to {temp_output_path}")

print("Processing complete. All combined surface data saved.")

Loading surface_temp_2013_combined.csv (Temp)...
Loading surface_temp_2014_combined.csv (Temp)...
Loading surface_temp_2015_combined.csv (Temp)...
Loading surface_temp_2016_combined.csv (Temp)...
Loading surface_temp_2017_combined.csv (Temp)...
Loading surface_temp_2018_combined.csv (Temp)...
Loading surface_temp_2019_combined.csv (Temp)...
Loading surface_temp_2020_combined.csv (Temp)...
Loading surface_temp_2021_combined.csv (Temp)...
Loading surface_temp_2022_combined.csv (Temp)...
Loading surface_temp_2023_combined.csv (Temp)...
Loading surface_temp_2024_combined.csv (Temp)...
Combined temp data saved to D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_yearlyfiles\surface_temp_all.csv
Processing complete. All combined surface data saved.


In [3]:
#When we originally received Live Ocean ROMS data, that data covered east of -124 degrees latitude. This covers all of our 
#sites except for two ones on the north coast (Pacific City) that just barely fall west of -124. So we had to go back in and 
#process a smaller versions of the ROMS dataset that covered that little easterly bit. That is this chunk of code.
#added February 2026
# Load dataset
# Do this once for the 2013-2019 data. Then go change the ds filename and the years to do it for the 2020-2024 years
ds = xr.open_dataset(r"D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/townsend3_surf_2020.01.01_2024.12.31.nc",
                    engine="netcdf4")
# Ensure ocean_time is in datetime format
ds['ocean_time'] = pd.to_datetime(ds['ocean_time'].values)
output_dir = "D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_datasets/"
os.makedirs(output_dir, exist_ok=True)  # Ensure output directory exists
years = [2020,2021,2022,2023,2024]  # OR USE [2013, 2014, 2015, 2016, 2017, 2018, 2019]
for year in years:
    for month in range(1, 13):  # Loop over months from January (1) to December (12)
        # Subset dataset for the specific year and month
        monthly_ds = ds.sel(ocean_time=(ds['ocean_time'].dt.year == year) & (ds['ocean_time'].dt.month == month))
        required_vars = [ 'hc', 'h', 'zeta', 'temp', 'lon_rho', 'lat_rho']
        if all(var in monthly_ds.variables for var in required_vars):
                      # Extract surface temperature (assuming -1 index gives the topmost layer)
            surf_temp = monthly_ds.temp[:, :, :].to_numpy()  # Extract top layer over time
            # Convert to xarray DataArray to preserve metadata
            surf_temp_da = xr.DataArray(surf_temp, dims=["ocean_time", "eta_rho", "xi_rho"], 
                                  coords={"ocean_time": monthly_ds.ocean_time, 
                                        "lat_rho": monthly_ds.lat_rho, 
                                        "lon_rho": monthly_ds.lon_rho},
                                name="surf_temp")
            temp_ds = xr.Dataset({
                "ocean_time": monthly_ds.ocean_time,
                "lat_rho": monthly_ds.lat_rho,
                "lon_rho": monthly_ds.lon_rho,
                "surf_temp": surf_temp_da
            })
            # Save datasets to disk
            temp_filename = f"{output_dir}ds_temp_east_{year}_{month:02d}.nc"
            temp_ds.to_netcdf(temp_filename)
            del temp_ds  # Free up memory 
#Ok step two, moving data out of .ncs and into csvs
# Define directories
input_dir = "D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_datasets"
output_dir = "D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_datasets_csvs"
# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)
# Define the years and months to loop over
months = range(1, 13)  # January (1) to December (12)
# Process each dataset one by one
for year in years:
    for month in months:
        # Update the filename for salt and temp datasets
        filename_temp = f"ds_temp_east_{year}_{month:02d}.nc"
        # Construct file paths for salt and temp datasets
        file_path_temp = os.path.join(input_dir, filename_temp)
        # Check if temp file exists and process it
        if os.path.exists(file_path_temp):
            print(f"Processing {filename_temp}...")
            ds_temp = xr.open_dataset(file_path_temp)
            df_temp = ds_temp.to_dataframe().reset_index()
            output_filename_temp = f"surface_temp_east_{year}_{month:02d}.csv"
            output_path_temp = os.path.join(output_dir, output_filename_temp)
            df_temp.to_csv(output_path_temp, index=False)
            ds_temp.close()
            del ds_temp, df_temp
            print(f"Saved {output_filename_temp}")
print("Processing complete. All surface data saved.")
##Ok Step 3: this should let you knit all the csvs together into one, which should be integrable with the larger surfaces dataframes
in_dir = "D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_datasets_csvs"
out_dir = "D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_yearlyfiles"
# Define the years to process
years = [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
# Loop through each year and combine salt and temp datasets separately
for year in years:
    # Get list of CSV files for salt and temp data
    temp_files = [f for f in os.listdir(in_dir) if f.startswith(f"surface_temp_east_{year}_") and f.endswith(".csv")]
    # Initialize lists to store dataframes
    df_temp_list = []
    # Load and append temp data
    for file in temp_files:
        file_path = os.path.join(in_dir, file)
        print(f"Loading {file} (Temp)...")
        df_temp_list.append(pd.read_csv(file_path))
    # Combine all temp data for the year and save
    if df_temp_list:
        df_temp_combined = pd.concat(df_temp_list, ignore_index=True)
        temp_output_path = os.path.join(out_dir, f"surface_temp_east_{year}_combined.csv")
        df_temp_combined.to_csv(temp_output_path, index=False)
        print(f"Combined temp data saved to {temp_output_path}")
print("Processing complete. All combined surface data saved.")

#and finally step 4:
# Define directories
data_dir = "D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_yearlyfiles"

# Define the years to process
years = [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

# Initialize lists to store dataframes for temp data
df_temp_list = []

# Loop through each year and load temp datasets
for year in years:
    # Get list of CSV files for salt and temp data
    temp_files = [f for f in os.listdir(data_dir) if f.startswith(f"surface_temp_east_{year}_") and f.endswith("combined.csv")]

    # Load and append temp data
    for file in temp_files:
        file_path = os.path.join(data_dir, file)
        print(f"Loading {file} (Temp)...")
        df_temp_list.append(pd.read_csv(file_path))

# Combine all temp data into one file and save
if df_temp_list:
    df_temp_combined = pd.concat(df_temp_list, ignore_index=True)
    temp_output_path = os.path.join(data_dir, "surface_temp_east_all.csv")
    df_temp_combined.to_csv(temp_output_path, index=False)
    print(f"Combined temp data saved to {temp_output_path}")

print("Processing complete. All combined surface data saved.")

#COde got hung up on this step for the east data. Start from here
#and finally step 4:
# Define directories
data_dir = "D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_yearlyfiles"

# Define the years to process
years = [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

# Initialize lists to store dataframes for temp data
df_temp_list = []

# Loop through each year and load temp datasets
for year in years:
    # Get list of CSV files for salt and temp data
    temp_files = [f for f in os.listdir(data_dir) if f.startswith(f"surface_temp_east_{year}_") and f.endswith("combined.csv")]

    # Load and append temp data
    for file in temp_files:
        file_path = os.path.join(data_dir, file)
        print(f"Loading {file} (Temp)...")
        df_temp_list.append(pd.read_csv(file_path))

# Combine all temp data into one file and save
if df_temp_list:
    df_temp_combined = pd.concat(df_temp_list, ignore_index=True)
    temp_output_path = os.path.join(data_dir, "surface_temp_east_all.csv")
    df_temp_combined.to_csv(temp_output_path, index=False)
    print(f"Combined temp data saved to {temp_output_path}")

print("Processing complete. All combined surface data saved.")

In [3]:
#Combine the two datasets together
#temp is 235,526,278 rows long
temp = pd.read_csv("D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_yearlyfiles/surface_temp_all.csv")
temp = temp.dropna()
#temp_east is 10,733,967 rows long
temp_east = pd.read_csv("D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_yearlyfiles/surface_temp_east_all.csv")
temp_combo = pd.concat([temp,temp_east])


In [3]:
temp_combo = pd.concat([temp,temp_east])
temp_combo

,ocean_time,eta_rho,xi_rho,surf_temp,lon_rho,lat_rho,Unnamed: 0
0,2013-01-01 12:00:00,0,0,11.086005,-124.996173,42.006750,NaN
1,2013-01-01 12:00:00,0,1,11.113357,-124.985495,42.006750,NaN
2,2013-01-01 12:00:00,0,2,11.118497,-124.974859,42.006750,NaN
3,2013-01-01 12:00:00,0,3,11.147481,-124.964265,42.006750,NaN
4,2013-01-01 12:00:00,0,4,11.157813,-124.953713,42.006750,NaN
...,...,...,...,...,...,...,...
10733962,2024-12-31 12:00:00,472,43,6.236618,-123.711971,46.298379,155486893.0
10733963,2024-12-31 12:00:00,472,44,6.143304,-123.705238,46.298379,155486894.0
10733964,2024-12-31 12:00:00,472,45,6.264662,-123.698505,46.298379,155486895.0
10733965,2024-12-31 12:00:00,472,46,6.274335,-123.691772,46.298379,155486896.0


In [4]:

temp_combo.to_csv("D:/ORKA_StatusReportFollowupPaper/LO_SurfaceBrookingsAreas/processed_yearlyfiles/surface_temp_all_combined.csv")

KeyboardInterrupt: 